# MLE Coding Challenges — Interactive Practice

The 20 most common ML engineering coding questions, each with a runnable implementation, test cases, and follow-up discussion.

**Format:** Problem → Implement → Test → Complexity Analysis → Follow-up Questions

## 1. Data Structures and Algorithms for ML

In [ ]:
import numpy as np
import heapq
from collections import defaultdict, deque
from typing import Optional

# ── Problem 1: Top-K Frequent Elements ──────────────────────────────────
# Given a list of predictions, return the K most common predicted classes.
# Relevance: batch inference label aggregation, recommendation top-K.

def top_k_frequent(elements: list, k: int) -> list:
    """Return k most frequent elements. O(n log k) using min-heap."""
    counts = defaultdict(int)
    for e in elements:
        counts[e] += 1
    # Min-heap of size k: pop smallest when heap exceeds k
    heap = []
    for elem, count in counts.items():
        heapq.heappush(heap, (count, elem))
        if len(heap) > k:
            heapq.heappop(heap)
    return [elem for _, elem in sorted(heap, reverse=True)]

print('Top-K Frequent Elements:')
preds = ['cat', 'dog', 'cat', 'cat', 'bird', 'dog', 'cat', 'bird', 'bird', 'dog']
print(f'  Predictions: {preds}')
print(f'  Top-2: {top_k_frequent(preds, 2)}')  # ['cat', 'dog']
print(f'  Time: O(n log k), Space: O(n + k)')
print()

# ── Problem 2: Sliding Window Average (streaming metrics) ───────────────
# Calculate running average of model loss over the last W batches.
# Relevance: training loss smoothing, online feature computation.

class SlidingWindowAverage:
    """O(1) update, O(1) query using deque and running sum."""
    def __init__(self, window_size: int):
        self.window = deque()
        self.window_size = window_size
        self.total = 0.0

    def add(self, value: float) -> float:
        self.window.append(value)
        self.total += value
        if len(self.window) > self.window_size:
            self.total -= self.window.popleft()
        return self.total / len(self.window)

losses = [0.5, 0.4, 0.35, 0.3, 0.4, 0.25, 0.2, 0.22]
smoother = SlidingWindowAverage(window_size=3)
smoothed = [smoother.add(loss) for loss in losses]
print('Sliding Window Average (w=3) — smoothing training loss:')
print(f'  Raw:      {[f"{l:.3f}" for l in losses]}')
print(f'  Smoothed: {[f"{l:.3f}" for l in smoothed]}')
print()

# ── Problem 3: Reservoir Sampling ───────────────────────────────────────
# Sample k items uniformly from a stream of unknown length.
# Relevance: streaming data sampling, online learning, large dataset subsampling.

def reservoir_sample(stream, k: int) -> list:
    """Uniformly sample k items from stream without knowing stream length."""
    import random
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            # Replace with probability k/(i+1)
            j = random.randint(0, i)
            if j < k:
                reservoir[j] = item
    return reservoir

np.random.seed(42)
stream = list(range(1000))
samples = [reservoir_sample(stream, 10) for _ in range(1000)]
avg_value = np.mean([np.mean(s) for s in samples])
print('Reservoir Sampling (k=10 from stream of 1000):')
print(f'  Expected mean ≈ 499.5, got: {avg_value:.1f}  ← unbiased!')
print(f'  Time: O(n), Space: O(k)  ← only stores k items')

## 2. ML-Specific Implementations

In [ ]:
import numpy as np
from typing import Callable

# ── Problem 4: K-Means from Scratch ─────────────────────────────────────

def kmeans(
    X: np.ndarray,
    k: int,
    max_iters: int = 100,
    tol: float = 1e-6,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """K-Means clustering. Returns (centroids, labels)."""
    rng = np.random.default_rng(seed)
    # Random initialization
    indices = rng.choice(len(X), k, replace=False)
    centroids = X[indices].copy()

    for iteration in range(max_iters):
        # Assignment step: assign each point to nearest centroid
        dists = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)  # (n, k)
        labels = np.argmin(dists, axis=1)

        # Update step: recompute centroids
        new_centroids = np.array([X[labels == j].mean(axis=0) for j in range(k)])

        # Convergence check
        if np.max(np.abs(new_centroids - centroids)) < tol:
            print(f'    Converged at iteration {iteration + 1}')
            break
        centroids = new_centroids

    return centroids, labels

# Test on synthetic clusters
np.random.seed(42)
X = np.vstack([
    np.random.normal([0, 0], 0.5, (100, 2)),
    np.random.normal([5, 5], 0.5, (100, 2)),
    np.random.normal([0, 5], 0.5, (100, 2)),
])
centroids, labels = kmeans(X, k=3)
print('K-Means (k=3):')
for i, c in enumerate(centroids):
    print(f'  Centroid {i}: [{c[0]:.2f}, {c[1]:.2f}]')
print(f'  Cluster sizes: {np.bincount(labels)}')
print()

# ── Problem 5: Gradient Descent ──────────────────────────────────────────

def gradient_descent(
    loss_fn: Callable,
    grad_fn: Callable,
    theta_init: np.ndarray,
    lr: float = 0.01,
    max_iters: int = 1000,
    tol: float = 1e-6,
) -> tuple[np.ndarray, list[float]]:
    """Vanilla gradient descent."""
    theta = theta_init.copy()
    losses = []
    for i in range(max_iters):
        loss = loss_fn(theta)
        losses.append(loss)
        grad = grad_fn(theta)
        theta -= lr * grad
        if np.max(np.abs(grad)) < tol:
            print(f'    Converged at iteration {i+1}, loss={loss:.6f}')
            break
    return theta, losses

# Fit linear regression y = 2x + 3 with GD
np.random.seed(42)
X_reg = np.random.randn(200, 1)
y_reg = 2 * X_reg.squeeze() + 3 + np.random.randn(200) * 0.5
X_b = np.column_stack([np.ones(200), X_reg])

mse_loss = lambda w: np.mean((X_b @ w - y_reg)**2)
mse_grad = lambda w: 2 / len(y_reg) * X_b.T @ (X_b @ w - y_reg)

w_gd, loss_history = gradient_descent(mse_loss, mse_grad, np.zeros(2), lr=0.1)
print('Gradient Descent — Linear Regression:')
print(f'  True: β0=3.0, β1=2.0')
print(f'  GD:   β0={w_gd[0]:.3f}, β1={w_gd[1]:.3f}')
print(f'  Initial loss: {loss_history[0]:.3f} → Final loss: {loss_history[-1]:.6f}')
print()

# ── Problem 6: Batch Normalization Forward Pass ───────────────────────────

def batch_norm_forward(
    X: np.ndarray,       # (batch, features)
    gamma: np.ndarray,   # (features,) scale
    beta: np.ndarray,    # (features,) shift
    eps: float = 1e-8,
) -> tuple[np.ndarray, dict]:
    """Batch normalization forward pass."""
    mu = X.mean(axis=0)                    # (features,)
    var = X.var(axis=0)                    # (features,)
    X_norm = (X - mu) / np.sqrt(var + eps) # normalize
    out = gamma * X_norm + beta            # scale and shift
    cache = {'X_norm': X_norm, 'mu': mu, 'var': var, 'gamma': gamma, 'eps': eps}
    return out, cache

X_bn = np.random.randn(32, 64)  # batch of 32, 64 features
gamma = np.ones(64); beta = np.zeros(64)
out, _ = batch_norm_forward(X_bn, gamma, beta)
print('Batch Normalization (batch=32, features=64):')
print(f'  Output mean ≈ 0: {out.mean(axis=0).mean():.6f}')
print(f'  Output std ≈ 1:  {out.std(axis=0).mean():.6f}')
print(f'  Purpose: Stabilizes training by normalizing layer inputs per batch')

## 3. Evaluation Metrics from Scratch

In [ ]:
import numpy as np

# ── Problem 7: ROC-AUC from Scratch ─────────────────────────────────────

def roc_auc(y_true: np.ndarray, y_score: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    """Compute ROC curve and AUC using the trapezoidal rule."""
    # Sort by descending score
    order = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[order]

    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos

    tprs, fprs = [0.0], [0.0]
    tp = fp = 0

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1
        tprs.append(tp / n_pos)
        fprs.append(fp / n_neg)

    tprs, fprs = np.array(tprs), np.array(fprs)
    auc = np.trapz(tprs, fprs)  # trapezoidal rule
    return fprs, tprs, auc

# ── Problem 8: Precision-Recall and Average Precision ────────────────────

def average_precision(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Compute AP (area under Precision-Recall curve)."""
    order = np.argsort(y_score)[::-1]
    y_sorted = y_true[order]
    tp_cumsum = np.cumsum(y_sorted)
    n_retrieved = np.arange(1, len(y_sorted) + 1)
    precisions = tp_cumsum / n_retrieved
    recalls = tp_cumsum / y_true.sum()
    # AP = sum of precision at each recall step where a positive is retrieved
    return np.sum(precisions[y_sorted == 1] * (1 / y_true.sum()))

# Test
np.random.seed(42)
n = 1000
y_true = np.random.binomial(1, 0.1, n)
y_score = y_true * np.random.beta(5, 2, n) + (1 - y_true) * np.random.beta(2, 5, n)

fprs, tprs, auc_val = roc_auc(y_true, y_score)
ap = average_precision(y_true, y_score)

# Verify against sklearn
from sklearn.metrics import roc_auc_score, average_precision_score
sklearn_auc = roc_auc_score(y_true, y_score)
sklearn_ap = average_precision_score(y_true, y_score)

print('Metrics from scratch vs sklearn:')
print(f'  ROC-AUC: ours={auc_val:.4f}, sklearn={sklearn_auc:.4f}  (match: {abs(auc_val-sklearn_auc)<1e-3})')
print(f'  Avg Precision: ours={ap:.4f}, sklearn={sklearn_ap:.4f}  (match: {abs(ap-sklearn_ap)<1e-3})')
print()

# ── Problem 9: Log Loss / Binary Cross-Entropy ──────────────────────────

def log_loss(y_true: np.ndarray, y_prob: np.ndarray, eps: float = 1e-15) -> float:
    """Binary cross-entropy loss."""
    y_prob = np.clip(y_prob, eps, 1 - eps)  # avoid log(0)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

# Edge cases
print('Log Loss edge cases:')
print(f'  Perfect predictions:  {log_loss(np.array([1,0,1]), np.array([0.999, 0.001, 0.999])):.6f}')
print(f'  Random predictions:   {log_loss(np.array([1,0,1]), np.array([0.5, 0.5, 0.5])):.6f}')
print(f'  Wrong predictions:    {log_loss(np.array([1,0,1]), np.array([0.001, 0.999, 0.001])):.6f}')
print()
print('Interview insight: AUC-ROC evaluates ranking quality; log loss evaluates calibration.')
print('Imbalanced classes (1% positives): use AUC-PR, not AUC-ROC!')

## 4. MLOps Implementations

In [ ]:
import numpy as np
from scipy import stats
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any

# ── Problem 10: Distribution Drift Detection ─────────────────────────────

def detect_drift(
    reference: np.ndarray,
    current: np.ndarray,
    method: str = 'ks',
    alpha: float = 0.05,
) -> dict:
    """Detect feature distribution drift between reference and current window."""
    if method == 'ks':
        stat, pvalue = stats.ks_2samp(reference, current)
        return {'drift': pvalue < alpha, 'stat': stat, 'pvalue': pvalue, 'method': 'KS'}
    elif method == 'psi':
        # Population Stability Index: > 0.25 = significant drift
        bins = np.percentile(reference, np.linspace(0, 100, 11))
        bins[0] -= 1e-10; bins[-1] += 1e-10
        ref_pct = np.histogram(reference, bins=bins)[0] / len(reference)
        cur_pct = np.histogram(current, bins=bins)[0] / len(current)
        ref_pct = np.where(ref_pct == 0, 1e-6, ref_pct)
        cur_pct = np.where(cur_pct == 0, 1e-6, cur_pct)
        psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
        return {'drift': psi > 0.25, 'psi': psi, 'method': 'PSI',
                'severity': 'high' if psi > 0.25 else 'moderate' if psi > 0.1 else 'low'}

np.random.seed(42)
ref = np.random.normal(50, 10, 5000)
no_drift = np.random.normal(50, 10, 1000)
with_drift = np.random.normal(65, 13, 1000)

print('Drift detection:')
for name, current in [('No drift', no_drift), ('With drift', with_drift)]:
    ks = detect_drift(ref, current, 'ks')
    psi = detect_drift(ref, current, 'psi')
    print(f'  {name}: KS p={ks["pvalue"]:.4f} ({"DRIFT" if ks["drift"] else "OK"}), PSI={psi["psi"]:.3f} ({psi["severity"]})')
print()

# ── Problem 11: Online Learning Rate Scheduler ───────────────────────────

def cosine_annealing_lr(
    step: int,
    total_steps: int,
    lr_max: float = 1e-3,
    lr_min: float = 1e-6,
    warmup_steps: int = 0,
) -> float:
    """Cosine annealing with optional linear warmup."""
    if step < warmup_steps:
        return lr_max * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

total = 1000
lrs = [cosine_annealing_lr(s, total, warmup_steps=100) for s in range(0, total+1, 100)]
print('Cosine annealing LR schedule (lr_max=1e-3, warmup=100 steps):')
for i, (step, lr) in enumerate(zip(range(0, total+1, 100), lrs)):
    bar = '█' * int(lr / 1e-3 * 20)
    print(f'  step {step:>4}: {lr:.2e}  {bar}')
print()

# ── Problem 12: Stratified K-Fold CV ────────────────────────────────────

def stratified_kfold(
    y: np.ndarray,
    n_splits: int = 5,
    shuffle: bool = True,
    seed: int = 42,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Stratified k-fold cross-validation split."""
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    fold_indices = [[] for _ in range(n_splits)]

    for cls in classes:
        cls_indices = np.where(y == cls)[0]
        if shuffle:
            rng.shuffle(cls_indices)
        for fold_i, chunk in enumerate(np.array_split(cls_indices, n_splits)):
            fold_indices[fold_i].extend(chunk.tolist())

    splits = []
    for fold_i in range(n_splits):
        val_idx = np.array(fold_indices[fold_i])
        train_idx = np.concatenate([np.array(fold_indices[j]) for j in range(n_splits) if j != fold_i])
        splits.append((train_idx, val_idx))
    return splits

y_imbalanced = np.array([0] * 900 + [1] * 100)
folds = stratified_kfold(y_imbalanced, n_splits=5)
print('Stratified 5-Fold (class ratio 90:10):')
for i, (train, val) in enumerate(folds):
    val_pos_rate = y_imbalanced[val].mean()
    print(f'  Fold {i+1}: val_size={len(val)}, positive_rate={val_pos_rate:.2f}  ← ~10% preserved')

## 5. System Design Coding: Feature Store and Pipelines

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Any, Optional
from collections import defaultdict
import time

# ── Problem 13: In-Memory Feature Store with TTL ─────────────────────────

@dataclass
class CachedFeature:
    value: Any
    expires_at: Optional[float]  # Unix timestamp

class FeatureStore:
    """Fast in-memory feature store with TTL eviction."""
    def __init__(self):
        self._store: dict[str, dict[str, CachedFeature]] = defaultdict(dict)
        self._stats = {'hits': 0, 'misses': 0, 'expired': 0}

    def set(self, entity_id: str, feature: str, value: Any, ttl_seconds: Optional[int] = None):
        expires_at = time.time() + ttl_seconds if ttl_seconds else None
        self._store[entity_id][feature] = CachedFeature(value=value, expires_at=expires_at)

    def get(self, entity_id: str, feature: str) -> Optional[Any]:
        entry = self._store.get(entity_id, {}).get(feature)
        if entry is None:
            self._stats['misses'] += 1
            return None
        if entry.expires_at and time.time() > entry.expires_at:
            del self._store[entity_id][feature]
            self._stats['expired'] += 1
            return None
        self._stats['hits'] += 1
        return entry.value

    def get_many(self, entity_id: str, features: list[str]) -> dict[str, Any]:
        return {f: self.get(entity_id, f) for f in features}

    @property
    def hit_rate(self) -> float:
        total = self._stats['hits'] + self._stats['misses']
        return self._stats['hits'] / total if total > 0 else 0.0

# Demo
fs = FeatureStore()
# Write user features
for user_id in range(100):
    fs.set(f'u{user_id}', 'age', np.random.randint(18, 65), ttl_seconds=3600)
    fs.set(f'u{user_id}', 'lifetime_value', np.random.exponential(100), ttl_seconds=3600)

# Simulate model serving: get features for 200 requests
for _ in range(200):
    uid = f'u{np.random.randint(0, 120)}'  # some cache misses for uid > 99
    fs.get_many(uid, ['age', 'lifetime_value'])

print('Feature Store performance:')
print(f'  Hit rate: {fs.hit_rate:.1%}')
print(f'  Stats: {fs._stats}')
print()

# ── Problem 14: Mini ETL Pipeline with Validation ────────────────────────

@dataclass
class ValidationResult:
    passed: bool
    errors: list[str] = field(default_factory=list)

def validate_feature_batch(records: list[dict]) -> ValidationResult:
    """Validate a batch of feature records before ingestion."""
    errors = []
    required_fields = {'user_id', 'age', 'total_spend', 'last_active'}

    for i, record in enumerate(records):
        missing = required_fields - set(record.keys())
        if missing:
            errors.append(f'Record {i}: missing fields {missing}')
            continue
        if not isinstance(record['user_id'], str):
            errors.append(f'Record {i}: user_id must be str')
        if not (0 <= record.get('age', -1) <= 150):
            errors.append(f'Record {i}: age {record["age"]} out of range [0, 150]')
        if record.get('total_spend', -1) < 0:
            errors.append(f'Record {i}: total_spend cannot be negative')

    return ValidationResult(passed=len(errors) == 0, errors=errors)

valid_batch = [
    {'user_id': 'u001', 'age': 28, 'total_spend': 500.0, 'last_active': '2024-01-15'},
    {'user_id': 'u002', 'age': 35, 'total_spend': 1200.0, 'last_active': '2024-01-14'},
]
invalid_batch = [
    {'user_id': 'u003', 'age': 200, 'total_spend': 100.0, 'last_active': '2024-01-13'},  # age out of range
    {'user_id': 'u004', 'total_spend': -50.0, 'last_active': '2024-01-12'},              # missing age, negative spend
]

print('ETL Validation:')
for name, batch in [('Valid', valid_batch), ('Invalid', invalid_batch)]:
    result = validate_feature_batch(batch)
    print(f'  {name}: passed={result.passed}')
    for err in result.errors:
        print(f'    ERROR: {err}')

## Exercises

1. **Implement DBSCAN from scratch:** Core algorithm — for each point, find ε-neighbors, label core/border/noise, then BFS/DFS to expand clusters. Test on a dataset with non-convex clusters where K-Means fails. Compare cluster quality using silhouette score.

2. **LRU Cache for model predictions:** Build an LRU (Least Recently Used) cache using a doubly linked list + hash map (no `functools.lru_cache`). Use it to cache top-K expensive model predictions. Measure hit rate and latency improvement.

3. **Mini MLflow — experiment tracker:** Implement a simple experiment tracker that logs: run metadata (timestamp, git hash), parameters (dict), metrics (dict with step), and artifacts (file paths). Support querying "best run by metric X" and comparing runs as a DataFrame.